# Dataset Preparation & Verification (`Aimpathy_AI_SignLanguage_Model`)

This notebook inspects and validates the isolated sign recordings and real continuous word recordings used for training and evaluating the ASL fingerspelling CTC pipeline.

## Directory Layout
* `data/takes` — Isolated letter/sign recordings used by `seq_data.py` to stitch synthetic fingerspelling sequences dynamically every epoch (2,477 files).
* `data/test` — Isolated sign recordings from unseen test signers for person-independent evaluation (486 files).
* `data/words_train` — Real continuous fingerspelled word takes recorded by training signers via `scripts/collect_words.py` (228 files).
* `data/words_test` — Held-out real continuous fingerspelled word takes (`leyu_test`, 91 takes across 30 words) used as the honest evaluation benchmark.

In [ ]:
from pathlib import Path
import sys

# Resolve project root whether launched from repo root or inside notebooks/
PROJECT_ROOT = Path.cwd().parent if Path.cwd().name == "notebooks" else Path.cwd()
SRC_DIR = PROJECT_ROOT / "src"
SCRIPTS_DIR = PROJECT_ROOT / "scripts"

if str(SRC_DIR) not in sys.path:
    sys.path.insert(0, str(SRC_DIR))

print(f"Project root : {PROJECT_ROOT}")
print(f"src path     : {SRC_DIR} (exists={SRC_DIR.exists()})")

## 1. Check Dataset Folder Counts & Vocabulary Configuration
Verify file counts across all four dataset splits and inspect `VOCAB`, `WORDS`, `FPS`, and `WINDOW_SEC` defined in `src/seq_data.py` and `src/utils.py`.

In [ ]:
import utils
import seq_data

splits = ["takes", "test", "words_train", "words_test"]
for split in splits:
    folder = PROJECT_ROOT / "data" / split
    if folder.exists():
        files = [f for f in folder.rglob("*") if f.is_file()]
        print(f"{split:12s} -> {len(files):4d} files")
    else:
        print(f"{split:12s} -> MISSING ({folder})")

print("\n--- Pipeline Constants ---")
print(f"Feature dim (FEAT_DIM) : {utils.FEAT_DIM}")
print(f"CTC Window (sec)       : {seq_data.WINDOW_SEC}s @ {seq_data.FPS} FPS (T={seq_data.T} frames)")
print(f"CTC Vocab ({len(seq_data.VOCAB)} tokens)  : {seq_data.VOCAB}")
print(f"Stitched Word List     : {len(seq_data.WORDS)} words")

## 2. Inspect Signer & Label Distribution
Run `scripts/check_data.py` or inspect `.npz` take filenames (`<label>_<signer>_<timestamp>.npz`) to verify signer separation between training (`takes`, `words_train`) and held-out evaluation (`test`, `words_test`).

In [ ]:
from collections import Counter

def summarize_npz_folder(folder_path: Path):
    npz_files = sorted(folder_path.rglob("*.npz"))
    labels = Counter()
    signers = Counter()
    for f in npz_files:
        parts = f.stem.split("_")
        if len(parts) >= 2:
            labels[parts[0]] += 1
            signers[parts[1]] += 1
    return len(npz_files), labels, signers

for split in splits:
    folder = PROJECT_ROOT / "data" / split
    if folder.exists():
        total, labels, signers = summarize_npz_folder(folder)
        print(f"=== {split} ({total} .npz files) ===")
        print("  Signers :", dict(signers))
        print("  Unique labels:", len(labels), "| Sample:", list(labels.items())[:8])

## 3. Run Official Data Integrity Check (`scripts/check_data.py`)
Execute the repository's built-in data validation script to verify that all required labels in `utils.ALL_LABELS` are accounted for without corrupted files.

In [ ]:
import subprocess

cmd = [sys.executable, str(SCRIPTS_DIR / "check_data.py")]
res = subprocess.run(cmd, cwd=PROJECT_ROOT, capture_output=True, text=True)
print(res.stdout)
if res.stderr:
    print("STDERR:\n", res.stderr)